# 3. Equivariant Message Passing

This is the most critical notebook in the MACE pedagogy. We will take a single MACE interaction block and mathematically prove that it is **Rotationally Equivariant**.

Equivariance means that if we rotate the input atomic geometry, the output features of the neural network will rotate by the exact same geometric transformation. i.e., $f(R \cdot x) = R \cdot f(x)$.

In [1]:
import os
import sys
import torch
import ase
from e3nn.o3 import rand_matrix

sys.path.append(os.path.abspath('..'))
from src.data import atoms_to_pyg_data
from src.basis import BesselBasis, SphericalHarmonicsBasis
from src.blocks import SimpleMACEBlock

### Setting up the Block and Data

We create a dummy water molecule and a single `SimpleMACEBlock`.

In [2]:
# 1. Dummy Data (H2O)
atoms = ase.Atoms('H2O', positions=[
    [0.0, 0.0, 0.0],
    [0.76, 0.58, 0.0],
    [-0.76, 0.58, 0.0]
])
data = atoms_to_pyg_data(atoms, cutoff=3.0)

# 2. Basis Functions
r_max = 3.0
num_radial = 4
l_max = 1  # Keep it simple: l=0 and l=1 only

radial_basis = BesselBasis(cutoff=r_max, num_radial=num_radial)
sh_basis = SphericalHarmonicsBasis(l_max=l_max)

# 3. Initial Node Features (Just ones for simplicity, pure scalars L=0)
node_dim = 4
node_feat = torch.ones((data.num_nodes, node_dim))
node_irreps = f"{node_dim}x0e"

# 4. MACE Block
block = SimpleMACEBlock(
    node_irreps=node_irreps,
    sh_irreps=str(sh_basis.irreps_out),
    radial_dim=num_radial
)

### Forward Pass 1: Original Orientation

We run the data through the block. Remember, the block internally calculates the Tensor Product between node features and the spherical harmonics of the edge vectors.

In [3]:
def forward_pass(pos, edge_index):
    row, col = edge_index
    edge_vec = pos[row] - pos[col]
    edge_len = torch.norm(edge_vec, dim=-1)
    
    radial_feat = radial_basis(edge_len)
    sh_feat = sh_basis(edge_vec)
    
    out_feat = block(node_feat, edge_index, radial_feat, sh_feat)
    return out_feat

out_orig = forward_pass(data.pos, data.edge_index)
print("Output shape:", out_orig.shape)

Output shape: torch.Size([3, 4])


### Forward Pass 2: Rotated Orientation

Now we apply a random 3D rotation matrix to the atomic coordinates and run the network again.

In [4]:
# Generate a random orthogonal 3x3 rotation matrix
rot_matrix = rand_matrix()
print("Random Rotation Matrix:")
print(rot_matrix)

# Rotate positions
rotated_pos = torch.einsum("ij,nj->ni", rot_matrix, data.pos)

# Forward pass on rotated coordinates
out_rotated = forward_pass(rotated_pos, data.edge_index)

Random Rotation Matrix:
tensor([[-4.0042e-01, -9.1608e-01, -2.1317e-02],
        [-5.4919e-02,  7.7112e-04,  9.9849e-01],
        [-9.1468e-01,  4.0099e-01, -5.0619e-02]])


### The Equivariance Proof

If the network is truly equivariant, we can take the output from the original pass (`out_orig`), mathematically rotate it using the same `rot_matrix` (using `e3nn`'s Irreps transform), and it should perfectly equal `out_rotated`!

In [5]:
from e3nn.o3 import Irreps

# Get the mathematical transformation operator (D-matrix) for our specific node irreps
D_matrix = Irreps(node_irreps).D_from_matrix(rot_matrix)

# Rotate the original output
out_orig_rotated = torch.einsum("ij,nj->ni", D_matrix, out_orig)

# Compare!
difference = torch.max(torch.abs(out_orig_rotated - out_rotated)).item()

print(f"Maximum difference between mathematically rotated output and network rotated output: {difference:.2e}")

if difference < 1e-5:
    print("\n✅ SUCCESS! The MACE Block is perfectly rotationally equivariant.")
else:
    print("\n❌ FAILED! Equivariance is broken.")

Maximum difference between mathematically rotated output and network rotated output: 1.19e-07

✅ SUCCESS! The MACE Block is perfectly rotationally equivariant.
